# Gradient descent on five students

A toy logistic regression: five students, hours studied, passed or not. The aim is to see every step of the maths on a problem small enough to check by hand.

1. Build the model and the loss.
2. Feel the slope by nudging (numerical gradient).
3. Walk downhill with gradient descent.
4. Compare with scikit-learn.
5. Check that the gradient formula `(p - y) x` matches the nudged slope.
6. Check the Hessian formula `p(1 - p) x x^T` the same way.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

hours  = np.array([1, 2, 3, 4, 5], dtype=float)
passed = np.array([0, 0, 1, 0, 1], dtype=float)

# centre the hours so the intercept means "a student with average hours"
x = hours - hours.mean()
y = passed
print("x =", x, "| y =", y)

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def loss(w, b):
    p = sigmoid(w * x + b)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print("loss at w=0, b=0:", round(loss(0, 0), 4), "(this is ln 2 = 0.6931, a coin flip for everyone)")

In [ ]:
def feel_slope(w, b, eps=1e-6):
    # nudge each knob a tiny bit and see how much the loss changes
    dw = (loss(w + eps, b) - loss(w - eps, b)) / (2 * eps)
    db = (loss(w, b + eps) - loss(w, b - eps)) / (2 * eps)
    return dw, db

print("slope at w=0, b=0:", np.round(feel_slope(0, 0), 4))

In [ ]:
w, b, step = 0.0, 0.0, 0.5
path = [(w, b, loss(w, b))]
for i in range(500):
    dw, db = feel_slope(w, b)
    w, b = w - step * dw, b - step * db
    path.append((w, b, loss(w, b)))
    if i == 0:
        print(f"after step 1: w={w:.4f}, b={b:.4f}, loss={loss(w, b):.4f}")

print(f"after 500 steps: w={w:.4f}, b={b:.4f}, loss={loss(w, b):.4f}")

In [ ]:
path = np.array(path)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(path[:, 2])
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].set_title("Walking downhill")

grid = np.linspace(-0.2, 1.6, 200)
ax[1].scatter(hours, passed, s=60, color="black", zorder=3, label="students")
xs = np.linspace(0.5, 5.5, 200)
ax[1].plot(xs, sigmoid(w * (xs - hours.mean()) + b), label="our model")
ax[1].set_xlabel("hours studied"); ax[1].set_ylabel("probability of passing")
ax[1].set_title("The model we built"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
lib = LogisticRegression(C=1e6).fit(x.reshape(-1, 1), y)
print(f"ours:         w={w:.4f}, b={b:.4f}")
print(f"scikit-learn: w={lib.coef_[0][0]:.4f}, b={lib.intercept_[0]:.4f}")

## Does the formula match the nudge?

Derived by hand: for one student, `dL/dw = (p - y) x` and `dL/db = p - y`. Over all five, take the average. Below, the formula and the nudge are compared at three different points.

In [ ]:
def formula_slope(w, b):
    p = sigmoid(w * x + b)
    return np.mean((p - y) * x), np.mean(p - y)

for (w0, b0) in [(0, 0), (1, -0.5), (-2, 1)]:
    print((w0, b0), "nudge:", np.round(feel_slope(w0, b0), 6), "| formula:", np.round(formula_slope(w0, b0), 6))

## Does the Hessian formula match?

The Hessian is the slope of the slope. By hand: entries `mean(p(1-p) x^2)`, `mean(p(1-p) x)`, `mean(p(1-p))`. Below, it is compared with nudging the formula slope.

In [ ]:
def hessian_formula(w, b):
    p = sigmoid(w * x + b)
    s = p * (1 - p)
    return np.array([[np.mean(s * x * x), np.mean(s * x)],
                     [np.mean(s * x),     np.mean(s)]])

def hessian_nudge(w, b, eps=1e-6):
    gw_plus  = np.array(formula_slope(w + eps, b)); gw_minus = np.array(formula_slope(w - eps, b))
    gb_plus  = np.array(formula_slope(w, b + eps)); gb_minus = np.array(formula_slope(w, b - eps))
    return np.column_stack([(gw_plus - gw_minus) / (2 * eps), (gb_plus - gb_minus) / (2 * eps)])

for (w0, b0) in [(0, 0), (1, -0.5)]:
    print((w0, b0))
    print("nudge:\n", np.round(hessian_nudge(w0, b0), 5))
    print("formula:\n", np.round(hessian_formula(w0, b0), 5))